<font face="Times New Roman" size=5>
<div dir=rtl align="center">

</font>
<br>
<img src="https://logoyar.com/content/wp-content/uploads/2021/04/sharif-university-logo.png" alt="University Logo" width="150" height="150">
<br>
<font face="Times New Roman" size=4 align=center>
Sharif University of Technology - Department of Electrical Engineering
</font>
<br>
<font face="Times New Roman" color="#008080" size=6>
Deep Generative Models
</font>
<hr/>
<font face="Times New Roman" color="#800080" size=5>
Assignment 1 : Deep Autoregressive Models
<br>
</font>
<font face="Times New Roman" size=4>
Instructor: Dr. S. Amini
<br>
<font face="Times New Roman" size=4>
TA: Iman Alizadeh Fakouri
<br>
</font>
<font size=4>
<font face="Times New Roman" size=3>
Fall 2026
<br>
</font>
<font face="Times New Roman" size=4>
</font>
<hr>
<font face="Times New Roman" size=4 align=center>
Feel free to ask your questions in Telegram : @imanf82
</font>
<br>
<hr>
</div></font>

# Question 1: Order-Agnostic MADE, Image Inpainting & Adversarial Robustness (40 Points)

Autoregressive models are exceptional at exact likelihood estimation because they model the joint distribution of data as a product of conditionals. In this question, you will build a Masked Autoencoder for Distribution Estimation (MADE). By selectively zeroing out specific weights, MADE transforms a standard, fully connected autoencoder into a rigorous density estimator that strictly obeys the autoregressive property.

To maximize the utility of your model, you will implement **Order-Agnostic Training**. Instead of relying on a single, fixed raster-scan order (like early PixelRNNs), your network will learn to handle *any* arbitrary topological ordering of the input dimensions. This property transforms the network into an ensemble of autoregressive models sharing a single set of parameters, allowing it to function as a highly flexible **Image Inpainter** and a robust **Anomaly Detector**.

## Part A: The Masked Linear Layer (5 Points)
In a standard multi-layer perceptron (MLP), every hidden unit connects to every input. This violates the autoregressive property, which dictates that the prediction for dimension $d$ must only depend on dimensions $< d$.

To enforce this strict sequential dependence without changing the dense architecture, we apply a binary mask $M^W$ to the weight matrix $W$ of a linear layer:
$$ h = \text{activation}((W \odot M^W)x + b) $$

Your first task is to implement a custom PyTorch linear layer that stores and applies this binary mask during the forward pass.

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
import torchvision.utils as vutils

torch.manual_seed(0)
np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

class MaskedLinear(nn.Linear):
    def __init__(self, in_features, out_features, bias=True):
        super(MaskedLinear, self).__init__(in_features, out_features, bias)
        self.register_buffer('mask', torch.ones(out_features, in_features))

    def set_mask(self, mask):
        self.mask.data.copy_(torch.from_numpy(mask.astype(np.uint8).T))

    def forward(self, input):
        ### TODO: Implement the masked forward pass (5 Points) ###
        # just multiply the weights by the mask before the linear op
        # (masked weights also get zero grad, so they never matter)
        return F.linear(input, self.weight * self.mask, self.bias)

## Part B: Order-Agnostic Training (10 Points)
Standard MADE models assign a fixed index to every input dimension and hidden unit. Connections are only permitted from lower indices to higher or equal indices, establishing a single directed acyclic graph (DAG).

In **Order-Agnostic Training**, the network is trained on a random permutation of the input ordering at every single parameter update. To implement this, your `MADE` class must include a mechanism to dynamically generate a valid topological ordering for a given random permutation, construct the corresponding binary masks ($M^W$) for every layer, and apply them.

By frequently resampling these masks during the training loop, the network learns to estimate the density under all possible factorization orders.

In [ ]:
class MADE(nn.Module):
    def __init__(self, input_size, hidden_sizes, num_masks=1):
        super(MADE, self).__init__()
        self.input_size = input_size
        self.hidden_sizes = hidden_sizes
        self.num_masks = num_masks   # I use this as the number of orderings to average over at test time

        self.net = nn.Sequential(
            MaskedLinear(input_size, hidden_sizes[0]),
            nn.ReLU(),
            MaskedLinear(hidden_sizes[0], hidden_sizes[1]),
            nn.ReLU(),
            MaskedLinear(hidden_sizes[1], input_size)
        )

        self.rng = np.random.RandomState(0)
        self.sample_masks()

    def sample_masks(self, order=None):
        ### TODO: Generate and apply Order-Agnostic binary masks for the network (10 Points) ###
        # order[i] = position of pixel i in the sequence (0 = first one).
        # if no order is given we draw a random permutation -> order agnostic training.
        # passing an order explicitly is needed later for inpainting.
        D = self.input_size
        L = len(self.hidden_sizes)
        if order is None:
            order = self.rng.permutation(D)
        order = np.asarray(order)

        # degree of each unit = how many inputs it is allowed to depend on
        m = {-1: order}
        for l in range(L):
            # sample between min of prev layer and D-2 (degree D-1 would connect to nothing)
            m[l] = self.rng.randint(m[l - 1].min(), D - 1, size=self.hidden_sizes[l])

        # hidden layers use >=, output layer uses strict < so output d never sees x_d itself
        masks = [m[l - 1][:, None] <= m[l][None, :] for l in range(L)]
        masks.append(m[L - 1][:, None] < m[-1][None, :])

        self.order = order
        layers = [layer for layer in self.net.modules() if isinstance(layer, MaskedLinear)]
        for layer, mask in zip(layers, masks):
            layer.set_mask(mask)   # masks are (in, out), set_mask transposes

    def forward(self, x):
        return self.net(x)   # logits of p(x_d = 1 | previous pixels)


def made_nll(model, x, n_orders=1, orders=None):
    # NLL per image (nats). With more than one ordering I average the likelihoods
    # (not the log-likelihoods), i.e. treat it as an ensemble -> logsumexp - log K
    model.eval()
    logps = []
    with torch.no_grad():
        for k in range(n_orders):
            model.sample_masks(None if orders is None else orders[k])
            logits = model(x)
            logps.append(-F.binary_cross_entropy_with_logits(logits, x, reduction='none').sum(1))
    logps = torch.stack(logps)
    return -(torch.logsumexp(logps, 0) - np.log(n_orders))


### TODO: Write the training loop to minimize Negative Log-Likelihood (NLL) on Binarized MNIST. ###
### Ensure you are dynamically resampling masks during training. Plot the training loss. ###
binarize = transforms.Compose([transforms.ToTensor(),
                               transforms.Lambda(lambda x: (x > 0.5).float().view(-1))])
bin_train = datasets.MNIST('./data', train=True, download=True, transform=binarize)
bin_test = datasets.MNIST('./data', train=False, download=True, transform=binarize)
bin_train_loader = torch.utils.data.DataLoader(bin_train, batch_size=100, shuffle=True)
bin_test_loader = torch.utils.data.DataLoader(bin_test, batch_size=1000, shuffle=False)

made = MADE(784, [1024, 1024], num_masks=8).to(device)
opt = torch.optim.Adam(made.parameters(), lr=1e-3)
EPOCHS_MADE = 30
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_MADE * len(bin_train_loader))

train_losses, test_nlls = [], []
for epoch in range(EPOCHS_MADE):
    made.train()
    t0 = time.time()
    for x, _ in bin_train_loader:
        x = x.to(device)
        made.sample_masks()   # new ordering every step
        logits = made(x)
        loss = F.binary_cross_entropy_with_logits(logits, x, reduction='sum') / x.size(0)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        train_losses.append(loss.item())
    # test nll with one random ordering, just to keep track
    nll = torch.cat([made_nll(made, x.to(device)) for x, _ in bin_test_loader]).mean().item()
    test_nlls.append(nll)
    print(f"epoch {epoch+1:2d}  train nll {np.mean(train_losses[-len(bin_train_loader):]):.2f}  "
          f"test nll {nll:.2f}  ({time.time()-t0:.0f}s)")

# fixed set of orderings so the numbers below are reproducible
eval_orders = [np.random.RandomState(100 + k).permutation(784) for k in range(made.num_masks)]
ens = torch.cat([made_nll(made, x.to(device), made.num_masks, eval_orders)
                 for x, _ in bin_test_loader]).mean().item()
print(f"test nll with {made.num_masks} orderings averaged: {ens:.2f} nats")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
smooth = np.convolve(train_losses, np.ones(100) / 100, mode='valid')
ax[0].plot(train_losses, alpha=0.3, label='per step'); ax[0].plot(np.arange(len(smooth)) + 99, smooth, label='moving avg')
ax[0].set_xlabel('iteration'); ax[0].set_ylabel('NLL (nats)'); ax[0].set_title('MADE train loss'); ax[0].set_ylim(60, 250); ax[0].legend()
ax[1].plot(range(1, EPOCHS_MADE + 1), test_nlls, 'o-'); ax[1].set_xlabel('epoch'); ax[1].set_ylabel('NLL (nats)'); ax[1].set_title('test NLL')
plt.tight_layout(); plt.show()

## Part C: Image Inpainting (10 Points)
Because your MADE model was trained to be order-agnostic, it has approximated the conditional distribution $p(x_{missing} | x_{observed})$ for any arbitrary subset of observed pixels. We can exploit this mathematical property to perform data imputation (inpainting).

To fill in missing pixels, we can query the model using an ordering where all *observed* pixels appear at the beginning of the autoregressive sequence, and all *missing* pixels appear at the end. By sampling the missing pixels sequentially, the model will generate highly probable completions based on the surrounding context.

You are provided with a batch of corrupted MNIST digits where the center 50% of the pixels have been entirely zeroed out. Your task is to mathematically formulate the correct ordering query, iteratively sample the missing dimensions, and reconstruct the corrupted digits.

In [ ]:
def inpaint_images(model, corrupted_images, missing_mask):
    """
    Inputs:
    - model: Trained Order-Agnostic MADE model
    - corrupted_images: Batch of images with missing pixels set to 0
    - missing_mask: Binary mask (1 indicates a missing pixel, 0 indicates an observed pixel)
    """
    model.eval()
    inpainted_images = corrupted_images.clone()

    with torch.no_grad():
        ### TODO: Implement the iterative inpainting algorithm to reconstruct the images (10 Points) ###
        shape = inpainted_images.shape
        x = inpainted_images.view(shape[0], -1)
        miss = missing_mask.reshape(-1).bool().cpu().numpy()
        obs_idx = np.where(~miss)[0]
        miss_idx = np.where(miss)[0]

        # put all observed pixels first (order among them doesn't matter) and the
        # missing ones at the end, so every missing pixel is conditioned on the whole observed part
        order = np.empty(x.size(1), dtype=np.int64)
        order[obs_idx] = np.random.permutation(len(obs_idx))
        order[miss_idx] = len(obs_idx) + np.arange(len(miss_idx))
        model.sample_masks(order)

        # sample the missing pixels one by one (zeros that are not filled yet can't
        # affect anything because they come later in the ordering)
        for d in miss_idx:
            p = torch.sigmoid(model(x)[:, d])
            x[:, d] = torch.bernoulli(p)

    return inpainted_images

### TODO: Load a test batch, corrupt the center 50%, run your inpainting algorithm, ###
### and visualize the original, corrupted, and reconstructed images side-by-side. ###
n_show = 10
originals = torch.stack([bin_test[i][0] for i in range(n_show)]).view(n_show, 1, 28, 28).to(device)

# center 20x20 block, 400/784 = ~51% of the image
missing_mask = torch.zeros(28, 28, device=device)
missing_mask[4:24, 4:24] = 1
corrupted = originals * (1 - missing_mask)

torch.manual_seed(1); np.random.seed(1)
recon1 = inpaint_images(made, corrupted, missing_mask)
recon2 = inpaint_images(made, corrupted, missing_mask)

show = corrupted.clone().repeat(1, 3, 1, 1); show[:, 0] += missing_mask * 0.5   # hole in red
rows = [originals.repeat(1, 3, 1, 1), show.clamp(0, 1), recon1.repeat(1, 3, 1, 1), recon2.repeat(1, 3, 1, 1)]
titles = ['original', 'corrupted (hole in red)', 'inpainted - sample 1', 'inpainted - sample 2']
fig, axes = plt.subplots(4, 1, figsize=(14, 7))
for ax, r, t in zip(axes, rows, titles):
    ax.imshow(vutils.make_grid(r.cpu(), nrow=n_show, padding=2, pad_value=1).permute(1, 2, 0))
    ax.set_title(t); ax.axis('off')
plt.tight_layout(); plt.show()

hole = missing_mask.bool().expand_as(originals)
print("pixel acc inside the hole:",
      f"{(recon1[hole] == originals[hole]).float().mean().item():.3f} / "
      f"{(recon2[hole] == originals[hole]).float().mean().item():.3f}")

## Part D: Adversarial Anomaly Detection (15 Points)
Discriminative classifiers learn decision boundaries, which can often be manipulated by pushing a sample slightly off the true data manifold using imperceptible noise. Generative models, conversely, evaluate the likelihood of the sample existing on the data manifold itself.

In this section, you will act as both the attacker and the defender.
First, you will implement the **Fast Gradient Sign Method (FGSM)** to attack a pre-trained standard CNN classifier:
$$x_{adv} = x + \epsilon \cdot \text{sign}(\nabla_x J(\theta, x, y))$$
This attack will generate adversarial examples that successfully fool the classifier into predicting the wrong digit.

Second, you will evaluate both the clean test images and your generated adversarial images using your trained MADE model. By plotting the Negative Log-Likelihood (NLL) distributions of both sets, you will analyze whether your autoregressive model can successfully distinguish the adversarial attacks based purely on their probability densities, acting as a security filter.

Run the cell below to load the target classifier `mnist_cnn_target.pth`.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.conv2_drop = nn.Dropout2d()
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2_drop(self.conv2(x)), 2))
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, training=self.training)
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
target_model = SimpleCNN().to(device)
target_model.load_state_dict(torch.load("mnist_cnn_target.pth", map_location=device))
target_model.eval()

def fgsm_attack(image, epsilon, data_grad):
    ### TODO: Implement the FGSM attack to generate an adversarial image (7 Points) ###
    perturbed = image + epsilon * data_grad.sign()
    return torch.clamp(perturbed, 0, 1)   # stay a valid image

### TODO: Generate adversarial samples, evaluate NLLs using your MADE model, ###
### and plot the likelihood histogram comparing clean vs. adversarial data (8 Points) ###
# the classifier takes plain [0,1] images (no normalization) - gets ~97% on test like that
gray_test = datasets.MNIST('./data', train=False, download=True, transform=transforms.ToTensor())
gray_loader = torch.utils.data.DataLoader(gray_test, batch_size=500, shuffle=False)

def make_adversarial(epsilon, n_batches=4):
    clean, adv, ys, pred_clean, pred_adv = [], [], [], [], []
    for b, (x, y) in enumerate(gray_loader):
        if b == n_batches: break
        x, y = x.to(device), y.to(device)
        x.requires_grad_(True)
        out = target_model(x)
        loss = F.nll_loss(out, y)
        target_model.zero_grad(); loss.backward()
        x_adv = fgsm_attack(x.detach(), epsilon, x.grad.detach())
        with torch.no_grad():
            pred_adv.append(target_model(x_adv).argmax(1))
        pred_clean.append(out.argmax(1).detach())
        clean.append(x.detach()); adv.append(x_adv); ys.append(y)
    return [torch.cat(t) for t in (clean, adv, ys, pred_clean, pred_adv)]

def soft_nll(x):
    # MADE was trained on binary images but the adversarial ones are grayscale.
    # if I threshold at 0.5 most of the small perturbation just disappears, so instead
    # I use the gray values directly as bernoulli targets (= expected NLL if we binarized randomly).
    # same thing for the clean images so the comparison is fair.
    x = x.view(x.size(0), -1)
    return made_nll(made, x, made.num_masks, eval_orders).cpu().numpy()

def auroc(neg, pos):
    # prob. that a random adversarial image has higher nll than a random clean one
    s = np.concatenate([neg, pos]); r = s.argsort().argsort() + 1
    return (r[len(neg):].sum() - len(pos) * (len(pos) + 1) / 2) / (len(neg) * len(pos))

epsilons = [0.05, 0.1, 0.2, 0.3]
results = {}
for eps in epsilons:
    clean, adv, ys, pc, pa = make_adversarial(eps)
    nll_c, nll_a = soft_nll(clean), soft_nll(adv)
    thr = np.percentile(nll_c, 95)   # threshold that lets 95% of clean images through
    results[eps] = dict(clean=nll_c, adv=nll_a, adv_imgs=adv, pa=pa, ys=ys)
    print(f"eps={eps:.2f}  acc {(pc == ys).float().mean():.3f} -> {(pa == ys).float().mean():.3f}  "
          f"nll clean {nll_c.mean():6.1f} / adv {nll_a.mean():6.1f}  "
          f"auroc {auroc(nll_c, nll_a):.3f}  detected at 5% fpr: {(nll_a > thr).mean():.3f}")

fig, axes = plt.subplots(1, len(epsilons), figsize=(18, 3.8))
for ax, eps in zip(axes, epsilons):
    r = results[eps]
    bins = np.linspace(min(r['clean'].min(), r['adv'].min()), max(np.percentile(r['clean'], 99.5), r['adv'].max()), 60)
    ax.hist(r['clean'], bins=bins, alpha=0.6, density=True, label='clean')
    ax.hist(r['adv'], bins=bins, alpha=0.6, density=True, label='adversarial')
    ax.set_title(f'eps = {eps}'); ax.set_xlabel('NLL (nats)'); ax.legend()
axes[0].set_ylabel('density')
plt.suptitle('MADE NLL, clean vs FGSM'); plt.tight_layout(); plt.show()

# some examples for eps = 0.1
r = results[0.1]
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(8):
    axes[0, i].imshow(gray_test[i][0][0], cmap='gray'); axes[0, i].set_title(f'label {r["ys"][i].item()}'); axes[0, i].axis('off')
    axes[1, i].imshow(r['adv_imgs'][i, 0].cpu(), cmap='gray'); axes[1, i].set_title(f'pred {r["pa"][i].item()}'); axes[1, i].axis('off')
plt.suptitle('FGSM, eps = 0.1'); plt.tight_layout(); plt.show()

# Question 2: Class-Conditional Gated PixelCNN & Receptive Field Analysis (60 Points)

While early PixelRNNs captured complex spatial dependencies using LSTMs, they suffered from extremely slow, sequential training. PixelCNNs solved this by using masked convolutions, allowing for highly parallelized training. However, vanilla masked convolutions suffer from a critical "blind spot" in their receptive field, causing the network to ignore a significant portion of the available image context.

In this question, you will implement the **Gated PixelCNN**, which elegantly resolves the blind spot using a two-stream architectural design. Furthermore, you will shift from unconditional generation to **Controllable Generation**. By injecting class embeddings directly into the network's feature maps, you will dictate exactly which digit the model synthesizes. To ensure your model trains rapidly on standard hardware, you will utilize a data quantization trick.

## Part A: Solving the Blind Spot (15 Points)
Standard $k \times k$ masked convolutions fail to capture the pixels located to the right of the current pixel in the rows strictly above it. As the network gets deeper, this blind spot grows linearly, leading to suboptimal density estimation.

To eliminate the blind spot, the PixelCNN decoder splits the convolution into two distinct streams:
1. **The Vertical Stack:** Conditions on all rows strictly above the current row. It is not allowed to see any pixels in the current row.
2. **The Horizontal Stack:** Conditions on the current row, strictly to the left of the current pixel.

Your task is to implement these custom convolutional layers. You must utilize padding and cropping to strictly enforce these receptive fields without allowing information to leak from the future.

In [ ]:
class VerticalStackConv(nn.Module):
    # vertical stack: output at row i can only use rows above i.
    # kernel is (k//2+1) x k and I pad only on top so it can't look down.
    #  - 'A' (first layer, sees the raw image): pad one extra row on top and drop the
    #    last output row -> everything shifts down by 1, so row i is not visible.
    #  - 'B': input features at row i already only depend on rows < i, so it's
    #    fine to include the current row.
    def __init__(self, in_channels, out_channels, kernel_size, mask_type='B'):
        super(VerticalStackConv, self).__init__()
        ### TODO: Implement the Vertical Convolution (7.5 Points) ###
        assert mask_type in ('A', 'B')
        self.mask_type = mask_type
        self.k = kernel_size
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=(kernel_size // 2 + 1, kernel_size))

    def forward(self, x):
        k = self.k
        H = x.size(2)
        if self.mask_type == 'A':
            x = F.pad(x, (k // 2, k // 2, k // 2 + 1, 0))   # left, right, top, bottom
            return self.conv(x)[:, :, :H, :]
        x = F.pad(x, (k // 2, k // 2, k // 2, 0))
        return self.conv(x)


class HorizontalStackConv(nn.Module):
    # horizontal stack: only the current row, pixels to the left.
    # same trick as above but along the width: 'A' = strictly left, 'B' = left + current
    def __init__(self, in_channels, out_channels, kernel_size, mask_type='B'):
        super(HorizontalStackConv, self).__init__()
        ### TODO: Implement the Horizontal Convolution (7.5 Points) ###
        assert mask_type in ('A', 'B')
        self.mask_type = mask_type
        self.k = kernel_size
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=(1, kernel_size // 2 + 1))

    def forward(self, x):
        k = self.k
        W = x.size(3)
        if self.mask_type == 'A':
            x = F.pad(x, (k // 2 + 1, 0, 0, 0))
            return self.conv(x)[:, :, :, :W]
        x = F.pad(x, (k // 2, 0, 0, 0))
        return self.conv(x)

## Part B: Gated Activation Units (10 Points)
Standard ReLUs are sufficient for standard computer vision tasks, but modeling complex multimodal distributions (like the precise intensity of a pixel) benefits from more expressive non-linearities.

Borrowing from the WaveNet architecture, Gated PixelCNN utilizes **Gated Activation Units**. The unit splits the feature maps and applies a hyperbolic tangent (acting as the primary feature representation) multiplied by a sigmoid (acting as a dynamic, learnable gate):
$$y = \tanh(W_{f} * x) \odot \sigma(W_{g} * x)$$

You will implement the `GatedPixelCNNBlock`, which integrates the vertical and horizontal convolutions, applies the gated activations, and maps the vertical stream into the horizontal stream using a $1 \times 1$ projection.

In [ ]:
class GatedPixelCNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, mask_type='B', num_classes=10):
        super(GatedPixelCNNBlock, self).__init__()

        self.v_conv = VerticalStackConv(in_channels, 2 * out_channels, kernel_size, mask_type)
        self.h_conv = HorizontalStackConv(in_channels, 2 * out_channels, kernel_size, mask_type)

        self.v_to_h = nn.Conv2d(2 * out_channels, 2 * out_channels, kernel_size=1)
        self.h_to_h = nn.Conv2d(out_channels, out_channels, kernel_size=1)

        ### TODO: Define the layer to project the class embedding into the feature maps (2 Points) ###
        # embedding size = num_channels = out_channels here; one projection per stack,
        # output is 2*C because it's added before splitting into tanh / sigmoid parts
        self.cond_v = nn.Linear(out_channels, 2 * out_channels, bias=False)
        self.cond_h = nn.Linear(out_channels, 2 * out_channels, bias=False)
        # no residual in the 'A' block, otherwise the output at (i,j) would see pixel (i,j)
        self.residual = (mask_type == 'B' and in_channels == out_channels)

    @staticmethod
    def gate(x):
        f, g = x.chunk(2, dim=1)
        return torch.tanh(f) * torch.sigmoid(g)

    def forward(self, v_x, h_x, class_embedding):
        ### TODO: Implement the two-stream forward pass with Gated Activation Units (8 Points) ###
        cv = self.cond_v(class_embedding)[:, :, None, None]   # same bias for every position
        ch = self.cond_h(class_embedding)[:, :, None, None]

        v = self.v_conv(v_x)
        v_out = self.gate(v + cv)

        # horizontal gets its own row + the vertical features (rows above), no shift
        # needed since my vertical stack already excludes the current row
        h = self.h_conv(h_x) + self.v_to_h(v)
        h_out = self.h_to_h(self.gate(h + ch))
        if self.residual:
            h_out = h_out + h_x
        return v_out, h_out

## Part C: Class Conditioning & 2-Bit Quantization (10 Points)
To steer the generation process, the network is conditioned on a class label $y$. You will pass $y$ through an `nn.Embedding` layer, which outputs a vector that acts as a class-dependent spatial bias. This embedding is added directly to the feature maps inside every `GatedPixelCNNBlock`.

**The Quantization Trick:** In the original PixelCNN paper, the network models all 256 possible pixel intensities using a massive 256-way softmax. This requires significant GPU memory and training time. To drastically accelerate your experiments, we will quantize the MNIST dataset to just **2 bits per pixel**, meaning there are only 4 possible color values (0, 1, 2, 3). Consequently, your final output layer must be configured as a 4-way classification map.

In [ ]:
class ConditionalPixelCNN(nn.Module):
    def __init__(self, in_channels=1, num_channels=64, num_layers=5, num_classes=10, color_levels=4,
                 kernel_size=3, first_kernel_size=7):
        super(ConditionalPixelCNN, self).__init__()
        self.in_channels = in_channels
        self.color_levels = color_levels

        self.embedding = nn.Embedding(num_classes, num_channels)

        ### TODO: Initialize the sequence of GatedPixelCNNBlocks (6 Points) ###
        # first block is type A, I gave it a 7x7 kernel so the receptive field is bigger from the start
        blocks = [GatedPixelCNNBlock(in_channels, num_channels, first_kernel_size, 'A', num_classes)]
        blocks += [GatedPixelCNNBlock(num_channels, num_channels, kernel_size, 'B', num_classes)
                   for _ in range(num_layers - 1)]
        self.blocks = nn.ModuleList(blocks)

        self.out_conv = nn.Conv2d(num_channels, color_levels * in_channels, kernel_size=1)

    def forward(self, x, labels):
        ### TODO: Implement the full forward pass returning logits of shape (Batch, color_levels, Height, Width) (4 Points) ###
        x = x.float() / (self.color_levels - 1) * 2 - 1   # 0..3 -> [-1, 1]
        emb = self.embedding(labels)
        v, h = x, x
        for block in self.blocks:
            v, h = block(v, h, emb)
        out = self.out_conv(F.relu(h))   # only the horizontal stack goes to the output
        B, _, H, W = out.shape
        if self.in_channels == 1:
            return out
        return out.view(B, self.color_levels, self.in_channels, H, W)

def get_2bit_mnist_loaders(batch_size=128):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: (x * 3).long())
    ])

    train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
    test_dataset = datasets.MNIST('./data', train=False, transform=transform)

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader

## Part D: Receptive Field Visualization via Autograd (10 Points)
If your Vertical and Horizontal stacks are padded or cropped incorrectly, the network will suffer from information leakage. Even a one-pixel overlap allows the model to "look into the future," rendering the density estimation entirely invalid.

You can verify the structural integrity of your masks using PyTorch's Autograd engine. By passing a dummy image with `requires_grad=True` into your initialized (but untrained) network, you can call `.backward()` on a specific spatial output. This automatically maps the gradient flow back to the input image, showing you exactly which input pixels influenced that specific prediction.

In [ ]:
def visualize_receptive_field(model, device, pos=(14, 14)):
    """
    Computes and plots the gradients of the input image with respect to a specific pixel's output
    to verify the integrity of the autoregressive masking.
    """
    model.eval()

    ### TODO: Pass a dummy image into the model, compute gradients for the output at spatial location (14, 14), ###
    ### and plot the absolute gradients to visually verify the receptive field. (10 Points) ###
    i, j = pos
    x = (torch.rand(1, 1, 28, 28, device=device) * 3).requires_grad_(True)
    labels = torch.zeros(1, dtype=torch.long, device=device)
    out = model(x, labels)
    out[0, :, i, j].sum().backward()
    grad = x.grad.abs()[0, 0].cpu()

    # the pixel itself and everything after it (raster order) should get exactly 0 gradient
    future = torch.zeros(28, 28, dtype=torch.bool)
    future[i, j:] = True
    future[i + 1:, :] = True
    leak = grad[future].sum().item()
    print(f"grad on current + future pixels: {leak:.3e}", "(ok)" if leak == 0 else "(LEAK)")
    print(f"pixels that affect ({i},{j}): {(grad[~future] > 0).sum().item()} out of {(~future).sum().item()} previous ones")

    fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
    ax[0].imshow(grad, cmap='viridis'); ax[0].set_title(f'|grad| for output ({i},{j})')
    rf = (grad > 0).float().numpy()
    rgb = np.stack([rf * 0.2, rf * 0.6, rf], -1); rgb[i, j] = [1, 0, 0]
    ax[1].imshow(rgb); ax[1].set_title('receptive field (blue), target pixel (red)')
    for a in ax:
        a.set_xticks(range(0, 28, 4)); a.set_yticks(range(0, 28, 4))
    plt.tight_layout(); plt.show()
    model.zero_grad()

# Initialize a dummy model and visualize the receptive field
dummy_model = ConditionalPixelCNN().to(device)
visualize_receptive_field(dummy_model, device)
# with more layers it grows, and there's no blind spot on the upper right
visualize_receptive_field(ConditionalPixelCNN(num_layers=12).to(device), device)

## Part E: Controllable Autoregressive Generation (15 Points)
Because this is a 4-way classification task per pixel, you will optimize your `ConditionalPixelCNN` using Standard Cross-Entropy Loss.

Once the model has converged, you will perform ancestral sampling. Unlike the parallelized training process, autoregressive generation is strictly sequential. You must pass the current state of the image into the network, sample the intensity for the next pixel from the output distribution, update the image, and repeat this for all $28 \times 28$ pixels.

Your final deliverable is a $10 \times 10$ image grid. Each row must contain generated images strictly conditioned on a specific digit class (e.g., Row 0 generates entirely '0's, Row 1 generates entirely '1's).

In [ ]:
### TODO: Write the training loop to minimize CrossEntropyLoss and report the final NLL in bits/dim (5 Points) ###
train_loader_2b, test_loader_2b = get_2bit_mnist_loaders(batch_size=128)

torch.manual_seed(0)
model = ConditionalPixelCNN(num_channels=64, num_layers=8).to(device)
print("params:", sum(p.numel() for p in model.parameters()))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS_PCNN = 4   # trained on cpu, more epochs / layers would help on a gpu
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_PCNN * len(train_loader_2b))
criterion = nn.CrossEntropyLoss()

def evaluate_bpd(model, loader):
    model.eval(); total, n = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            total += F.cross_entropy(model(x, y), x[:, 0], reduction='sum').item(); n += x.numel()
    return total / n / np.log(2)   # nats -> bits

pcnn_losses, pcnn_test_bpd = [], []
for epoch in range(EPOCHS_PCNN):
    model.train(); t0 = time.time()
    for x, y in train_loader_2b:
        x, y = x.to(device), y.to(device)
        logits = model(x, y)               # (B, 4, 28, 28)
        loss = criterion(logits, x[:, 0])  # targets (B, 28, 28) with values 0..3
        optimizer.zero_grad(); loss.backward(); optimizer.step(); scheduler.step()
        pcnn_losses.append(loss.item() / np.log(2))
    pcnn_test_bpd.append(evaluate_bpd(model, test_loader_2b))
    print(f"epoch {epoch+1}  train {np.mean(pcnn_losses[-len(train_loader_2b):]):.4f}  "
          f"test {pcnn_test_bpd[-1]:.4f} bits/dim  ({time.time()-t0:.0f}s)")

print(f"\nfinal test nll: {pcnn_test_bpd[-1]:.4f} bits/dim  ({pcnn_test_bpd[-1] * 784:.1f} bits per image, "
      f"uniform would be 2 bits/dim)")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(pcnn_losses, alpha=0.4, label='train')
ax.plot(np.arange(1, EPOCHS_PCNN + 1) * len(train_loader_2b), pcnn_test_bpd, 'o-', label='test')
ax.set_xlabel('iteration'); ax.set_ylabel('bits/dim'); ax.set_ylim(0, 0.6); ax.legend(); ax.set_title('PixelCNN loss')
plt.tight_layout(); plt.show()


def generate_conditional_images(model, device):
    """
    Autoregressively sample from the trained conditional PixelCNN.
    """
    model.eval()
    samples = torch.zeros(100, 1, 28, 28).to(device)
    labels = torch.arange(10).repeat_interleave(10).to(device)

    with torch.no_grad():
        ### TODO: Implement the autoregressive pixel-by-pixel generation loop (10 Points) ###
        for i in range(28):
            for j in range(28):
                logits = model(samples, labels)[:, :, i, j]
                probs = F.softmax(logits, dim=1)
                samples[:, 0, i, j] = torch.multinomial(probs, 1).squeeze(1).float()

    ### TODO: Plot the final 10x10 grid using torchvision.utils.make_grid or matplotlib ###
    grid = vutils.make_grid(samples.cpu() / 3, nrow=10, padding=2, pad_value=0.5)
    plt.figure(figsize=(8, 8))
    plt.imshow(grid.permute(1, 2, 0)); plt.axis('off')
    plt.title('samples, row k = digit k')
    plt.show()

    return samples

torch.manual_seed(0)
samples = generate_conditional_images(model, device)